# OpenBMB VoxCPM2 — Kaggle production demo

This notebook is the Kaggle demo qualification thin production-demo surface.

Required Kaggle settings:

- Accelerator: **GPU T4 x2** for the full qualification matrix.
- Attach Kaggle model: **dangkhoa2016/openbmb-voxcpm2**.
- Internet: enabled during source/dependency bootstrap.
- Inference itself is forced into local/offline model mode after bootstrap.

The notebook creates a new workspace on every run and does not reuse prior virtual environments, logs, model downloads, or project checkouts from `/kaggle/working`.

Select a canonical execution-profile qualification execution profile with the `VOXCPM_DEMO_PROFILE` environment variable: `cpu`, `cuda-single`, `cuda-replica`, or `auto`. Default: `auto`.


In [ ]:
from pathlib import Path
import os

PROFILE = os.environ.get("VOXCPM_DEMO_PROFILE", "auto")
PROJECT_REF = os.environ.get("VOXCPM_DEMO_REF", "main")
WORKSPACE = Path(
    os.environ.get(
        "VOXCPM_DEMO_WORKSPACE",
        f"/kaggle/working/voxcpm2-demo-{PROFILE}",
    )
)
REPO_URL = "https://github.com/dangkhoa2016/OpenBMB-VoxCPM2-Inference.git"

assert PROFILE in {"cpu", "cuda-single", "cuda-replica", "auto"}
print("profile=", PROFILE)
print("project_ref=", PROJECT_REF)
print("workspace=", WORKSPACE)


In [ ]:
import json
import shutil
import subprocess
import sys

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True)

SOURCE = WORKSPACE / "source"
UPSTREAM = WORKSPACE / "upstream"
VENV = WORKSPACE / "venv"

def run(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)

run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(SOURCE)])
run(["git", "fetch", "--depth=1", "origin", PROJECT_REF], cwd=SOURCE)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=SOURCE)

resolved_sha = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=SOURCE, text=True
).strip()
print("resolved_project_sha=", resolved_sha)

lock = json.loads((SOURCE / "provenance/voxcpm-source-lock.json").read_text())
upstream_repo = lock["upstream_code"]["repository"]
upstream_commit = lock["upstream_code"]["commit"]

run(["git", "clone", "--filter=blob:none", "--no-checkout", upstream_repo, str(UPSTREAM)])
run(["git", "fetch", "--depth=1", "origin", upstream_commit], cwd=UPSTREAM)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=UPSTREAM)

run([
    sys.executable, "-m", "venv",
    "--without-pip", "--system-site-packages", str(VENV),
])
VENV_PY = VENV / "bin/python"

run([str(VENV_PY), "-m", "pip", "install", "-q", "-e", f"{SOURCE}[api]", "--no-deps"])
run([str(VENV_PY), "-m", "pip", "install", "-q", "-e", str(UPSTREAM), "--no-deps"])
run([
    str(VENV_PY), "-m", "pip", "install", "-q",
    "fastapi==0.136.1",
    "uvicorn[standard]==0.46.0",
    "httpx==0.28.1",
    "addict",
    "wetext",
    "modelscope",
    "funasr",
    "spaces",
    "argbind",
    "datasets>=3,<4",
    "gradio>=6,<7",
])

run([
    str(VENV_PY), "-c",
    "import torch, voxcpm; "
    "print('torch=', torch.__version__); "
    "print('cuda=', torch.version.cuda, torch.cuda.is_available(), torch.cuda.device_count()); "
    "print('voxcpm=', getattr(voxcpm, '__version__', None))",
])


In [ ]:
import json
import os
import subprocess

offline_env = os.environ.copy()
offline_env.update({
    "VOXCPM_OFFLINE": "1",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
})

verified = subprocess.check_output(
    [str(VENV_PY), "scripts/verify_model.py"],
    cwd=SOURCE,
    env=offline_env,
    text=True,
)
model_report = json.loads(verified)
assert model_report["status"] == "ok"
assert model_report["offline"] is True
assert model_report["remote_acquisition_count"] == 0
print(json.dumps({
    "status": model_report["status"],
    "model_id": model_report["model_id"],
    "source_kind": model_report["source_kind"],
    "offline": model_report["offline"],
    "remote_acquisition_count": model_report["remote_acquisition_count"],
}, indent=2, sort_keys=True))


In [ ]:
import os
import subprocess

EVIDENCE_DIR = WORKSPACE / "evidence"
EVIDENCE_DIR.mkdir()

run_env = os.environ.copy()
run_env.update({
    "VOXCPM_OFFLINE": "1",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
})

run([
    str(VENV_PY),
    "scripts/kaggle_production_demo.py",
    "--profile", PROFILE,
    "--workspace", str(EVIDENCE_DIR),
], cwd=SOURCE, env=run_env)


In [ ]:
import json

evidence_path = EVIDENCE_DIR / f"kaggle-{PROFILE}-evidence.json"
evidence = json.loads(evidence_path.read_text())

assert evidence["git_sha"] == resolved_sha
assert evidence["profile"] == PROFILE
assert evidence["request"]["status_code"] == 200
assert evidence["request"]["sample_rate"] == 48000
assert evidence["request"]["channels"] == 1
assert evidence["cleanup"]["gpu_processes_match_baseline"] is True
assert evidence["cleanup"]["port_open"] is False

print(json.dumps({
    "git_sha": evidence["git_sha"],
    "profile": evidence["profile"],
    "resolved_profile": evidence["resolved_profile"],
    "startup": evidence["startup"],
    "request": evidence["request"],
    "cleanup": evidence["cleanup"],
}, indent=2, sort_keys=True))
print("KAGGLE_KAGGLE_RUN_ALL=PASS")
print("evidence_path=", evidence_path)
